# Week 0 — Foundations of MLOps

### Why models rot, and the engineering discipline that keeps them alive

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain what MLOps is *as a discipline* and how it differs from DevOps.
2. Identify the four sources of non-reproducibility in an ML system.
3. Build a content-addressable hashing utility to fingerprint code, data, and environment.
4. Pin and capture a reproducible environment manifest from scratch.
5. Reason about the ML lifecycle as a directed graph of artifacts.

## Prerequisites

- Comfortable Python (functions, classes, dataclasses).
- Basic NumPy.
- Having trained at least one model before (any of the other courses in this profile suffice).

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_t3a_6x1p


## 1. 🧠 What is MLOps, really?

DevOps answers: *how do we ship and operate software reliably?* The software is
deterministic — same input, same output. MLOps inherits all of that and then
adds a brutal twist: **the behaviour of the system is defined by data you do not
fully control, and that data changes.**

A traditional program is `f(x)` where `f` is written by a human. An ML system is
`f_θ(x)` where `θ` was *learned* from a dataset `D`. So the real artifact you are
operating is not the code — it is the triple:

$$ \text{System} = (\;\text{code},\; \text{data } D,\; \text{environment } E\;) \;\longrightarrow\; \theta \longrightarrow f_\theta $$

Change *any* of `code`, `D`, or `E` and you can get a different `θ`, and therefore
a different system — often silently. MLOps is the engineering discipline that
makes this triple **explicit, versioned, testable, observable, and recoverable.**

> **The one-sentence definition we'll use all course:** MLOps is the practice of
> treating the *(code, data, environment) → model → prediction* pipeline as a
> first-class, versioned, reproducible, and monitored software artifact.

### 1.1 The MLOps maturity ladder

It helps to know where a team is. We'll keep returning to this ladder.

| Level | Name | What it looks like |
|------|------|--------------------|
| 0 | Manual / notebook | Train in a notebook, copy weights to a server by hand. No tracking. |
| 1 | Reproducible | Pinned envs, versioned data + code, experiments logged. |
| 2 | Automated training | A pipeline retrains on new data on demand or on a schedule. |
| 3 | Automated deployment | CI/CD ships validated models automatically; rollbacks exist. |
| 4 | Full CD + monitoring | Drift triggers retraining; the loop closes itself. |

This course walks you up this ladder one week at a time. Week 0 gets you off
level 0 by attacking **reproducibility** — the foundation everything else stands on.

## 2. 🧠 The four horsemen of non-reproducibility

When a colleague says *"it works on my machine but the metric is different on
yours"*, the cause is always one of four things:

1. **Code drift** — different source than you think (uncommitted changes, wrong branch).
2. **Data drift** — a different dataset, or the "same" dataset that quietly changed.
3. **Environment drift** — different library versions, BLAS backend, CUDA, OS.
4. **Stochastic drift** — unseeded randomness (init, shuffling, dropout, data order).

We'll build a small tool that nails down all four. The unifying idea is the
**content hash**: if two things have the same hash, they are bit-for-bit identical.

## 3. 🔧 From scratch: content-addressable fingerprinting

Git, Docker, DVC, and model registries are all built on one primitive: hashing
content to get a stable address. Let's build that primitive ourselves so it is
never mysterious again.

In [2]:
def hash_bytes(b: bytes) -> str:
    """Stable 12-char content fingerprint. SHA-256 truncated for readability."""
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj: Any) -> str:
    """Hash any JSON-serialisable object deterministically.

    The trick that bites everyone: dict key order. json.dumps(sort_keys=True)
    makes {'a':1,'b':2} and {'b':2,'a':1} hash identically, as they should.
    """
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    """Hash a numpy array by its raw bytes + shape + dtype."""
    payload = a.tobytes() + str(a.shape).encode() + str(a.dtype).encode()
    return hash_bytes(payload)

# Sanity checks
print("same dict, different order ->", hash_obj({"a": 1, "b": 2}), hash_obj({"b": 2, "a": 1}))
print("different values          ->", hash_obj({"a": 1}), hash_obj({"a": 2}))
arr = np.arange(12).reshape(3, 4)
print("array fingerprint         ->", hash_array(arr))
print("same array again          ->", hash_array(np.arange(12).reshape(3, 4)))

same dict, different order -> 43258cff783f 43258cff783f
different values          -> 015abd7f5cc5 7e8059f49558
array fingerprint         -> f45abfabc14c
same array again          -> f45abfabc14c


Notice the second pair of dict hashes differ while the first pair match. That is
the whole game: **identity by content, not by reference.** A model registry uses
exactly this to dedupe identical models; Docker uses it to cache layers; Git uses
it for every commit.

## 4. 🔧 From scratch: capturing the environment

`E` (the environment) is the most-forgotten horseman. Let's capture it the way
`pip freeze` does — but understand every byte.

In [3]:
import platform, importlib.metadata as ilm

def capture_environment() -> dict:
    """Snapshot enough of the environment to reproduce a run."""
    pkgs = {}
    for dist in ilm.distributions():
        name = dist.metadata["Name"]
        if name:
            pkgs[name.lower()] = dist.version
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "machine": platform.machine(),
        # Only keep a few high-signal packages for display brevity
        "key_packages": {k: pkgs.get(k) for k in ["numpy", "nbformat", "pip"]},
        "n_packages_total": len(pkgs),
    }

env = capture_environment()
print(json.dumps(env, indent=2))
print("\nEnvironment fingerprint:", hash_obj(env))

{
  "python": "3.12.3",
  "platform": "Linux-6.18.5-x86_64-with-glibc2.39",
  "machine": "x86_64",
  "key_packages": {
    "numpy": "2.4.4",
    "nbformat": "5.10.4",
    "pip": "24.0"
  },
  "n_packages_total": 204
}

Environment fingerprint: f316639e6d2e


The environment fingerprint is a single string that changes the instant *any*
pinned dependency changes. Store it next to every experiment and "works on my
machine" becomes a falsifiable claim: compare the two fingerprints.

## 5. 🔧 From scratch: taming stochasticity

Randomness is not the enemy — *unrecorded* randomness is. The fix is to make the
seed an explicit input and to seed **every** RNG your stack touches.

In [4]:
def seed_everything(seed: int = 0) -> int:
    """Seed all RNGs we can reach. Returns the seed so you can log it.

    In a real project you'd also seed torch / tf / cuda here. We seed what's
    installed and note the pattern; the point is that the seed is a *logged input*,
    never an implicit default buried in a library.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    return seed

s = seed_everything(42)
a = np.random.rand(3)
seed_everything(42)
b = np.random.rand(3)
print("Reproducible draw:", np.allclose(a, b), "(seed logged as", s, ")")

Reproducible draw: True (seed logged as 42 )


## 6. 🔧 From scratch: the *run manifest* — pulling it all together

A **run manifest** is the artifact that makes a training run reproducible. It is
just a dict containing the fingerprints of all four horsemen. Save it with every
run and you can always answer: *what exactly produced this model?*

In [5]:
@dataclass
class RunManifest:
    run_id: str
    seed: int
    code_hash: str
    data_hash: str
    env_hash: str
    params: dict
    created_at: float = field(default_factory=time.time)

    @property
    def reproducibility_key(self) -> str:
        """Two runs with the same key MUST produce the same model."""
        return hash_obj({
            "seed": self.seed,
            "code": self.code_hash,
            "data": self.data_hash,
            "env": self.env_hash,
            "params": self.params,
        })

def make_manifest(source_code: str, data: np.ndarray, params: dict, seed: int) -> RunManifest:
    return RunManifest(
        run_id=hash_obj({"t": time.time(), "r": random.random()}),
        seed=seed,
        code_hash=hash_bytes(source_code.encode()),
        data_hash=hash_array(data),
        env_hash=hash_obj(capture_environment()),
        params=params,
    )

# Simulate two "training runs" of a trivial model and prove reproducibility.
SOURCE = "theta = data.mean(axis=0)"   # pretend this is our training code
data = np.random.RandomState(7).rand(100, 3)
params = {"lr": 0.01, "epochs": 10}

m1 = make_manifest(SOURCE, data, params, seed=42)
m2 = make_manifest(SOURCE, data, params, seed=42)

print("run 1 id:", m1.run_id)
print("run 2 id:", m2.run_id, "(different id, same recipe)")
print("repro key 1:", m1.reproducibility_key)
print("repro key 2:", m2.reproducibility_key)
print("\nSame recipe -> same key?", m1.reproducibility_key == m2.reproducibility_key)

run 1 id: 0c6bf67a62b5
run 2 id: d00d9d321134 (different id, same recipe)
repro key 1: 62690262486b
repro key 2: 62690262486b

Same recipe -> same key? True


Two runs have **different run IDs** (they happened at different times) but the
**same reproducibility key** because code, data, env, params, and seed all match.
If someone changes one byte of the training code, the key changes, and you have a
machine-checkable signal that the model *should* be retrained. This single idea
underlies model registries, pipeline caching, and CI gates — all of which we
build in the coming weeks.

## 7. 🧠 The ML lifecycle as an artifact graph

Step back. Everything we just hashed is a **node** in a graph, and the arrows are
*"was produced from"* relationships:

```
 raw_data ──┐
            ▼
        features ──► model ──► evaluation ──► deployment ──► predictions
            ▲          ▲                                          │
          code       env                                         ▼
                                                            monitoring
                                                                  │
                                                  (drift) ────────┘  feeds back to retrain
```

MLOps is, concretely, the job of making this graph **explicit, versioned, and
automatable**. Each remaining week of the course owns one part of this graph:

- **Week 1** — `model` node: experiment tracking + registry.
- **Week 2** — packaging the `model` + `env` into a portable unit.
- **Week 3** — turning the unit into a live `predictions` service.
- **Week 4** — the arrows: CI/CD that validates `code → model → deployment`.
- **Week 5** — `monitoring` and the feedback arrow (drift).
- **Week 6** — the special pain of doing all of the above for LLMs.

## 8. 🏭 In practice

What you just built by hand maps directly onto real tools:

| You built | Real-world equivalent | What it adds |
|-----------|----------------------|--------------|
| `hash_array`, `hash_bytes` | Git / DVC / Docker content hashing | Distributed dedup, delta storage |
| `capture_environment` | `pip freeze`, `conda env export`, Docker images | OS-level isolation, exact pinning |
| `seed_everything` | framework seeding utilities | Covers CUDA/cuDNN determinism flags |
| `RunManifest` | MLflow run / W&B run / metadata stores | UI, search, team sharing, lineage |

You are now off **maturity level 0**. The rest of the course is about climbing
the ladder without ever losing the reproducibility property you just secured.

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Extend `RunManifest` to also hash the *order* of the training data rows. Show that shuffling the data (without reseeding) changes the data hash — and argue whether that *should* change the reproducibility key.
2. Write `diff_manifests(m1, m2)` that returns which of the four horsemen differ between two runs. This is the core of a 'why are my metrics different?' debugger.
3. `capture_environment` currently truncates to a few packages. Make it hash the *full* dependency set, then deliberately upgrade one package and confirm the env hash changes.
4. Argue for or against: 'A run ID should be the reproducibility key, not a timestamp-based hash.' What breaks in each design?

---

## ✅ Recap — Week 0

- MLOps operates the *(code, data, environment) → model → prediction* triple, not just code.
- Non-reproducibility always reduces to four horsemen: code, data, environment, stochasticity.
- Content hashing is the universal primitive — we built it and it reappears in every tool.
- A run manifest with a reproducibility key turns 'works on my machine' into a checkable claim.
- The ML lifecycle is an artifact graph; each remaining week owns one part of it.

### Coming up next

**Week 1 — Experiment Tracking & Model Registry.** We'll build a working, file-backed tracking server and registry from scratch (your own mini-MLflow), then see exactly what MLflow adds on top.